# ZedProfiler Walkthrough
This notebook serves as a walkthrough of the ZedProfiler codebase, demonstrating how to use the various functions and features for image analysis and feature extraction. 
The notebook is organized into sections corresponding to different types of analyses, such as granularity, intensity, colocalization, neighbors, and texture. Each section includes example code snippets that illustrate how to call the relevant functions with appropriate parameters.

## Imports

In [1]:
import logging
import os
import pathlib

import numpy as np
import pandas as pd
import tifffile

from zedprofiler.featurization.colocalization import compute_colocalization
from zedprofiler.featurization.granularity import compute_granularity
from zedprofiler.featurization.intensity import compute_intensity
from zedprofiler.featurization.neighbors import compute_neighbors
from zedprofiler.featurization.texture import compute_texture
from zedprofiler.featurization.volumesizeshape import compute_volume_size_shape
from zedprofiler.IO.loading_classes import (
    ImageSetConfig,
    ImageSetLoader,
    ObjectLoader,
    TwoObjectLoader,
)

logging.basicConfig(level=logging.INFO)  # ← moved to the end

annotation=NoneType required=False default_factory=list


## Define Paths and Parameters

In [2]:
# use your own data directory or generate synthetic data for testing?
GENERATE_SYNTHETIC_DATA = True

In [3]:
if not GENERATE_SYNTHETIC_DATA:
    # put your own data directories here if you want to test with real data
    image_set_path = pathlib.Path(
        os.path.expanduser(
            "~/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/zstack_images/C4-2"
        )
    ).resolve()
    label_set_path = pathlib.Path(
        os.path.expanduser(
            "~/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/segmentation_masks/C4-2"
        )
    ).resolve()

    # mapping to unique keys in the image names to extract
    # what that given image is
    channel_mapping = {
        "DNA": 405,
        "ER": 488,
        "AGP": 555,
        "Mito": 640,
        "Nuclei": "nuclei_",
        "Cytoplasm": "cytoplasm_",
        "Cell": "cell_",
        "Organoid": "organoid_",
    }

    CHANNEL1 = "DNA"
    CHANNEL2 = "ER"
    COMPARTMENT = "Nuclei"
else:
    notebook_root = pathlib.Path.cwd().resolve()
    image_set_path = (notebook_root / "test_data" / "dummy_image_set").resolve()
    label_set_path = (notebook_root / "test_data" / "dummy_label_set").resolve()
    channel_mapping = {
        "Channel1": "channel1",
        "Channel2": "channel2",
        "Nuclei": "nuclei_labels",
    }
    CHANNEL1 = "Channel1"
    CHANNEL2 = "Channel2"
    COMPARTMENT = "Nuclei"
    if not image_set_path.is_dir() or not label_set_path.is_dir():
        # generate and save dummy data if real data is not available
        channel1_array = (np.random.rand(100, 100, 10) * 255).astype(np.uint8)
        channel2_array = (np.random.rand(100, 100, 10) * 255).astype(np.uint8)
        nuclei_labels = np.random.randint(0, 5, size=(100, 100, 10), dtype=np.uint16)
        image_set_path.mkdir(parents=True, exist_ok=True)
        label_set_path.mkdir(parents=True, exist_ok=True)
        tifffile.imwrite(image_set_path / "channel1.tif", channel1_array)
        tifffile.imwrite(image_set_path / "channel2.tif", channel2_array)
        tifffile.imwrite(label_set_path / "nuclei_labels.tif", nuclei_labels)

## Loading Objects and Image Sets

In [4]:
isc = ImageSetConfig(
    image_set_name="test_image_set",
    raw_image_key_name=[CHANNEL1, CHANNEL2],
    label_key_name=[COMPARTMENT],
)

image_set_loader = ImageSetLoader(
    anisotropy_spacing=[1, 0.1, 0.1],
    channel_mapping=channel_mapping,
    image_set_path=image_set_path,
    label_set_path=label_set_path,
    config=isc,
    # we can also directly pass arrays in
    # this is the way into ZedProfiler when the images are multichannel
    # load the image arrays in and pass them in directly instead
    # of loading from path
    # future work will include expanding support for other file types
    # we currently use bioio as the default for loading arrays
    # most biomedical imaging formats should be supported by bioio
    image_set_array=None,
    label_set_array=None,
)

In [5]:
ol = ObjectLoader(
    image_set_loader=image_set_loader,
    channel_name=CHANNEL1,
    compartment_name=COMPARTMENT,
)

In [6]:
coloc_loader = TwoObjectLoader(
    image_set_loader=image_set_loader,
    compartment=COMPARTMENT,
    channel1=CHANNEL1,
    channel2=CHANNEL2,
)

## Colocalization

In [7]:
colocalization_features = compute_colocalization(
    two_object_loader=coloc_loader,
    thr=15,
    fast_costes="Accurate",
    channel1=CHANNEL1,
    channel2=CHANNEL2,
)

## Granularity

In [8]:
granularity_features = compute_granularity(
    object_loader=ol,
    radius=10,  # radius of the structuring element for background
    # removal (CellProfiler default)
    granular_spectrum_length=16,  # range of the granular spectrum
    subsample_size=0.25,  # subsample the image for faster processing
    image_sample_size=0.25,  # further subsample for background removal
    mask_threshold=0.9,  # threshold for determining mask after interpolation
    verbose=False,  # print out intermediate steps and values for debugging
)

## Intensity

In [9]:
intensity_features = compute_intensity(
    object_loader=ol,
)

## Neighbors

In [10]:
neighbors_features = compute_neighbors(
    object_loader=ol,
    distance_threshold=10,
    anisotropy_factor=image_set_loader.anisotropy_spacing[0]
    / image_set_loader.anisotropy_spacing[
        1
    ],  # z to xy spacing ratio for distance calculation
)

## Texture

In [11]:
texture_features = compute_texture(
    object_loader=ol,
    distance=3,
    grayscale=256,
)

## Volume, Size, and Shape

In [12]:
volume_size_shape_features = compute_volume_size_shape(
    image_set_loader=image_set_loader,
    object_loader=ol,
)

## All features in a DataFrame

In [13]:
coloc_df = pd.DataFrame(colocalization_features)
granularity_df = pd.DataFrame(granularity_features)
intensity_df = pd.DataFrame(intensity_features)
neighbors_df = pd.DataFrame(neighbors_features)
texture_df = pd.DataFrame(texture_features)
volume_size_shape_df = pd.DataFrame(volume_size_shape_features)
# subtract 2 from each df shape to account for image_set and object_id
# columns that will be used for merging
size_of_all_dfs = [
    df.shape[1] - 2
    for df in [
        coloc_df,
        granularity_df,
        intensity_df,
        neighbors_df,
        texture_df,
        volume_size_shape_df,
    ]
]
# add the 2 back to account for the image_set and object_id columns
# that will be retained in the final merged DataFrame
expected_num_columns = sum(size_of_all_dfs) + 2

In [ ]:
columns_to_merge = ["Metadata_Experiment_ImageSet", "Metadata_Object_ObjectID"]
for df in [
    coloc_df,
    granularity_df,
    intensity_df,
    neighbors_df,
    texture_df,
    volume_size_shape_df,
]:
    if not all(col in df.columns for col in columns_to_merge):
        raise ValueError(
            f"DataFrame is missing required columns for merging: {columns_to_merge}"
        )
# merge all features into a single DataFrame
all_features_df = pd.merge(
    coloc_df,
    pd.merge(
        granularity_df,
        pd.merge(
            intensity_df,
            pd.merge(
                neighbors_df,
                pd.merge(
                    texture_df, volume_size_shape_df, on=columns_to_merge, how="inner"
                ),
                on=columns_to_merge,
                how="inner",
            ),
            on=columns_to_merge,
            how="inner",
        ),
        on=columns_to_merge,
        how="inner",
    ),
    on=columns_to_merge,
    how="inner",
)
if all_features_df.shape[1] != expected_num_columns:
    raise ValueError(
        f"""
        Merged DataFrame has {all_features_df.shape[1]} columns
        but expected {expected_num_columns} based on individual DataFrames"""
    )
all_features_df

,Metadata_Object_ObjectID,Metadata_Experiment_ImageSet,Nuclei_Channel1-Channel2_Colocalization_Correlation,Nuclei_Channel1-Channel2_Colocalization_MandersCoeffM1,Nuclei_Channel1-Channel2_Colocalization_MandersCoeffM2,Nuclei_Channel1-Channel2_Colocalization_OverlapCoeff,Nuclei_Channel1-Channel2_Colocalization_MandersCoeffCostesM1,Nuclei_Channel1-Channel2_Colocalization_MandersCoeffCostesM2,Nuclei_Channel1-Channel2_Colocalization_RankWeightedColocalizationCoeff1,Nuclei_Channel1-Channel2_Colocalization_RankWeightedColocalizationCoeff2,...,Nuclei_Channel1_VolumeSizeShape_MinX,Nuclei_Channel1_VolumeSizeShape_MaxX,Nuclei_Channel1_VolumeSizeShape_MinY,Nuclei_Channel1_VolumeSizeShape_MaxY,Nuclei_Channel1_VolumeSizeShape_MinZ,Nuclei_Channel1_VolumeSizeShape_MaxZ,Nuclei_Channel1_VolumeSizeShape_Extent,Nuclei_Channel1_VolumeSizeShape_EulerNumber,Nuclei_Channel1_VolumeSizeShape_EquivalentDiameter,Nuclei_Channel1_VolumeSizeShape_SurfaceArea
0,1,test_image_set,0.007289,0.847344,0.848077,1.168619,0.0,0.0,0.719243,0.719405,...,0,10,0,100,0,100,0.19946,-4347,33.647469,946.328965
1,2,test_image_set,0.007289,0.847344,0.848077,1.168619,0.0,0.0,0.719243,0.719405,...,0,10,0,100,0,100,0.19947,-4278,33.648031,946.328965
2,3,test_image_set,0.007289,0.847344,0.848077,1.168619,0.0,0.0,0.719243,0.719405,...,0,10,0,100,0,100,0.20026,-4455,33.692393,946.328965
3,4,test_image_set,0.007289,0.847344,0.848077,1.168619,0.0,0.0,0.719243,0.719405,...,0,10,0,100,0,100,0.20234,-4557,33.808641,946.328965


In [15]:
expected_num_compartments = 4
expected_num_of_channels = 4
expected_num_of_total_features = (
    expected_num_columns - 2
)  # subtract 2 for image_set and object_id columns
expected_num_of_total_features = (
    expected_num_of_total_features
    * expected_num_compartments
    * expected_num_of_channels
    + 2
)
expected_num_of_total_features

3698

In [16]:
if all_features_df.shape[1] != expected_num_columns:
    raise ValueError(
        f"Expected {expected_num_columns} columns but got {all_features_df.shape[1]}"
    )